# Static arguments in jit: windowed pooling

`jax.jit` traces every argument as an abstract array by default. That is a problem when an argument decides a **shape** (a window size) or the **structure** of the computation (a string choosing a branch): a tracer has no concrete value to use. Mark such arguments static with `static_argnames`, and JAX compiles one version per distinct value.

Write `pool_sequence(x, pool, mode)`, compiled with `jax.jit` (fix the decorator in the starter):

- `x`: `(T, D)` float array.
- `pool`: int $\ge 1$, the window length.
- `mode`: `'mean'` or `'max'`.

Window $j$ covers rows $[j \cdot \text{pool},\ \min((j+1)\cdot\text{pool},\ T))$, so there are $\lceil T / \text{pool} \rceil$ windows. Return a float32 array of shape `(ceil(T / pool), D)` holding each window's mean or max over its rows. When `T` is not a multiple of `pool`, the last window is shorter and uses **only its real rows** (no padding values leak into the mean or max).

No Python loop over windows.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

@jax.jit                     # pool sets a shape and mode picks a branch: this decorator needs to know that
def pool_sequence(x, pool, mode):
    """x (T, D). Return (ceil(T / pool), D) window means or maxes."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[0.0, 1.0], [2.0, 3.0], [4.0, 5.0], [6.0, 7.0]], "dtype": "float64"}, 2, "mean"], "e": {"__nd__": [[1.0, 2.0], [5.0, 6.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[1.0], [5.0], [2.0], [-1.0], [3.0]], "dtype": "float64"}, 2, "max"], "e": {"__nd__": [[5.0], [2.0], [3.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[1.0], [5.0], [2.0], [-1.0], [3.0]], "dtype": "float64"}, 2, "mean"], "e": {"__nd__": [[3.0], [0.5], [3.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[-1.0531157544867582, -1.776491303816993, -2.5532918384570134], [-0.13796506137840808, -1.0137194090532766, -1.3521418253819912], [-0.6537883844162056, -1.4971178525878377, -0.289957591366348], [-0.5512671317684119, -0.17873768757050404, -1.073858701475369], [-0.8466289662382713, -0.37958424600772894, -0.5801952016057006], [-1.2715513764583872, -1.2923865934033114, -1.7987863384903786], [-0.02607383754457069, -1.3837097563119558, -0.9058431408224087]], "dtype": "float64"}, 3, "max"], "e": {"__nd__": [[-0.13796506822109222, -1.0137194395065308, -0.28995758295059204], [-0.551267147064209, -0.17873768508434296, -0.5801951885223389], [-0.026073837652802467, -1.3837097883224487, -0.9058431386947632]], "dtype": "float32"}}, {"a": [{"__nd__": [[-0.8163147296909071, 0.08130305629403443, 0.2814308365081419], [-1.5989883258714874, -1.730724189576384, 0.3550784031579345], [-0.8621880728268713, 1.2083570546510172, 0.39454619699169213], [0.30718460927779667, 0.2084426281897281, 0.8954750922624016], [-0.1530461454323981, 1.085630989593339, -0.5259143195557643], [-0.22499771732365406, -0.6805051952715889, 0.5976618821818257], [0.035884245959506046, -0.8622670569283692, 2.0018799427117075]], "dtype": "float64"}, 3, "mean"], "e": {"__nd__": [[-1.0924971103668213, -0.14702138304710388, 0.3436851501464844], [-0.023619752377271652, 0.20452278852462769, 0.3224075436592102], [0.03588424623012543, -0.8622670769691467, 2.001879930496216]], "dtype": "float32"}}, {"a": [{"__nd__": [[-0.3637140287393912, -0.11629977362405139, 0.3941647724606624, 1.7476509117779921], [-0.14838363341709068, 0.7031832470096712, 0.09879533762985714, -1.46379366686345], [0.7919772598652602, 1.0206392828537416, -0.23998627535552336, -0.015436337317648078], [-1.5013161339395207, -1.52295197049763, -0.39151994178010335, 0.6259333764593479], [-0.46957460601575257, 0.29202287772434254, 0.9677673497209652, 0.09576239436548745]], "dtype": "float64"}, 1, "max"], "e": {"__nd__": [[-0.3637140393257141, -0.11629977077245712, 0.3941647708415985, 1.7476508617401123], [-0.14838363230228424, 0.7031832337379456, 0.0987953394651413, -1.4637936353683472], [0.7919772863388062, 1.0206393003463745, -0.23998627066612244, -0.01543633732944727], [-1.5013161897659302, -1.5229519605636597, -0.3915199339389801, 0.6259333491325378], [-0.46957460045814514, 0.2920228838920593, 0.9677673578262329, 0.09576239436864853]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.5744683559978767, 1.447873507557578, -1.2460281198874508, 0.7440750749018286], [-1.8846846024583004, 0.4316425810503539, 0.20299464141371132, -0.5433347654582601], [-1.0773130472080232, -0.06183105779539427, -0.4102386596807664, 0.0976215963890434], [-0.9078017428322972, 0.5455273351229474, -1.0358502464384105, -0.3042352740012662], [-0.39198805527117014, 1.2678874917895713, -1.495693806222624, -0.41402382825701145]], "dtype": "float64"}, 8, "mean"], "e": {"__nd__": [[-0.7374638319015503, 0.7262200713157654, -0.7969632148742676, -0.08397945016622543]], "dtype": "float32"}}, {"a": [{"__nd__": [[1.1824606639063542, -0.6583412380249722, -1.2394711292950078, 0.28284148850071195, -0.29003040792375917, 0.8347549227982314], [-0.9203901507043266, 0.19091921538999582, 0.666127318168495, 1.035721840408728, -1.106145239973971, 1.4237887646772474], [-0.9006966533747327, -1.512550548496178, -0.9545692035185357, 0.15484928437706327, -0.3217840988429495, 0.4404556188056906], [0.49560327678002974, 0.3181445011036235, -0.254527904507233, 1.8140779814742283, -2.21603354078873, -1.7343490436198923], [-0.5068859522845482, -1.2552172711707161, -2.5559253523683503, -0.7950685249971639, -0.7492610878085537, -2.448549330978423], [0.8586929389625256, 1.299112064743398, 1.234301117381651, -0.5083406871217644, 0.14761216667290408, -1.1208102361387011], [0.20643731067377746, -0.6741753557015862, 0.4098832168893899, -0.3429722884134346, 1.0337574293500387, 0.826607747842839], [1.2389447067916677, -1.605295746346106, -1.8196072145661006, 0.4047423226792462, -0.5771601777932914, 0.3978289091024805], [0.29888085906307704, 0.18021493482187, 0.9192535355148113, 0.5566165137700867, 0.4724460241355843, -0.8938198675949263], [1.7246652903995292, 0.12661360551217557, 1.1070492973573018, 0.3270671245431495, 1.269365442389993, 0.15353532836324615], [0.21761066540615967, 0.137813376639258, -0.7086793387845165, -0.5720164743644298, -0.14694883314805018, -0.1169814348105986], [-0.9826943136535353, 2.356566943505155, -0.9813704058581482, -0.3059986194929355, 2.445287314867692, -0.6596879191716871], [0.7659382611041363, 0.6608838229283897, -0.20802063893841533, -1.4289290277038424, -1.5530243823807452, -0.007917216985390385], [0.4208160808202212, -2.1561194677615996, 0.1528481310312397, -0.6541230464920597, 0.8952039565608904, 0.17636171653962981], [-0.8014308325233171, -0.4235352083760813, 0.04848512986884152, 1.349816415581546, 3.050979726276049, 0.3664154919980895], [0.13002931950925808, 0.7127132273295342, 0.04592363650402586, 0.7104279161602328, 0.469049123586706, 0.08745628790171407]], "dtype": "float64"}, 4, "max"], "e": {"__nd__": [[1.1824606657028198, 0.3181445002555847, 0.6661273241043091, 1.8140779733657837, -0.29003041982650757, 1.4237887859344482], [1.23894464969635, 1.29911208152771, 1.2343010902404785, 0.4047423303127289, 1.0337574481964111, 0.8266077637672424], [1.7246652841567993, 2.356566905975342, 1.1070493459701538, 0.556616485118866, 2.4452872276306152, 0.15353532135486603], [0.7659382820129395, 0.7127132415771484, 0.15284812450408936, 1.3498164415359497, 3.0509796142578125, 0.36641550064086914]], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(pool_sequence, json.loads(_TESTS), 'jax', *[1e-05, 1e-06])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")